# Evaluation: Base vs. Fine-tuned

Side-by-side ROUGE-1/2/L + BERTScore-F1 on the SAMSum test split. This is the notebook to point to when an interviewer asks 'did your fine-tune actually help, and how do you know?'.

**What we compare:**
1. **Zero-shot baseline**: `Llama-3.2-3B-Instruct` with the same system prompt, no fine-tuning.
2. **LoRA fine-tuned**: same base + the adapter trained in `01_finetune_samsum_lora.ipynb`.

**Why both ROUGE and BERTScore?**
- ROUGE measures n-gram overlap. Fast, well-understood, the standard in summarization papers — but it punishes valid paraphrases.
- BERTScore uses a BERT model to compute semantic similarity per-token. It captures "the model said the same thing differently" cases that ROUGE misses.
- Using both is the modern convention. If they disagree, that's a signal worth investigating.

In [ ]:
import sys
sys.path.insert(0, '..')
import json
from pathlib import Path

## 1. Baseline (zero-shot)

Run this once; it takes ~5 min on T4 for 100 samples.

In [ ]:
!cd .. && python -m src.eval --n_samples 100 --output ./outputs/eval_baseline.json

## 2. Fine-tuned

In [ ]:
!cd .. && python -m src.eval \
    --adapter_dir ./outputs/llama32-samsum-lora \
    --n_samples 100 \
    --output ./outputs/eval_finetuned.json

## 3. Compare

In [ ]:
baseline = json.loads(Path('../outputs/eval_baseline.json').read_text())['metrics']
finetuned = json.loads(Path('../outputs/eval_finetuned.json').read_text())['metrics']

print(f'{'Metric':<15} {'Baseline':<12} {'Fine-tuned':<12} {'Δ':<10}')
print('-' * 50)
for k in ['rouge1_f', 'rouge2_f', 'rougeL_f', 'bertscore_f1']:
    b, f = baseline[k], finetuned[k]
    delta = f - b
    arrow = '↑' if delta > 0 else '↓'
    print(f'{k:<15} {b:<12.4f} {f:<12.4f} {arrow} {abs(delta):.4f}')

## 4. Inspect failure cases

Look at examples where fine-tuned ROUGE is lower than baseline — sometimes the model produced a valid summary that just didn't match the reference's wording. This is where BERTScore tends to be more forgiving than ROUGE.

In [ ]:
ft = json.loads(Path('../outputs/eval_finetuned.json').read_text())
for s in ft['samples'][:3]:
    print('=== DIALOGUE ===')
    print(s['dialogue'][:300] + '...')
    print('=== REFERENCE ===')
    print(s['reference'])
    print('=== PREDICTION ===')
    print(s['prediction'])
    print()